# Praktik Week 5: Pengenalan YOLOv8 & Roboflow

Notebook ini dirancang untuk memandu Anda melakukan praktik pertama dengan **YOLOv8** dan mempersiapkan integrasi dataset dari **Roboflow**. Kita akan belajar:
1. Instalasi library pendukung (`ultralytics` dan `roboflow`).
2. Menjalankan inferensi deteksi objek dasar menggunakan model YOLOv8 pre-trained pada Gambar.
3. Menjalankan inferensi YOLOv8 pada Video & Real-Time Webcam.
4. Mengintegrasikan dataset kustom dari Roboflow.
5. Menjalankan pengujian (testing) model hasil training pada berkas video asli.
6. Melakukan evaluasi kuantitatif metrik performa model (Precision, Recall, mAP).

## Langkah 1: Instalasi Library Pendukung

Kita perlu menginstal library `ultralytics` yang memuat semua fungsi YOLOv8, serta library `roboflow` untuk mengunduh dataset yang telah kita beri label di cloud secara terprogram.

In [ ]:
# Jalankan perintah ini untuk menginstal dependensi
!pip install ultralytics roboflow opencv-python matplotlib

## Langkah 2: Import Library & Verifikasi Instalasi

Setelah instalasi selesai, kita pastikan library dapat di-import dengan benar dan memeriksa versi Ultralytics yang terinstal.

In [1]:
import ultralytics
from ultralytics import YOLO
import cv2 as cv
import matplotlib.pyplot as plt
import os

print("Ultralytics Version:", ultralytics.__version__)
print("Setup berhasil! YOLOv8 siap digunakan.")

Ultralytics Version: 8.4.65
Setup berhasil! YOLOv8 siap digunakan.


## Langkah 3: Inferensi Dasar dengan Pre-trained Model YOLOv8

Kita akan menggunakan model **YOLOv8 Nano (yolov8n)**. Model ini sangat ringan (~6MB) dan cepat untuk dijalankan di laptop.

Secara default, saat kita pertama kali menjalankan model dengan bobot `yolov8n.pt`, library Ultralytics akan mendeteksi apakah file bobot tersebut sudah ada di folder kerja kita. Jika belum ada, sistem akan mengunduhnya secara otomatis dari server Ultralytics (bobot pre-trained dilatih menggunakan dataset **COCO** yang memiliki 80 kelas objek umum seperti orang, mobil, anjing, bus, dll.).

In [2]:
# 1. Load pre-trained model YOLOv8 Nano
model = YOLO('yolov8n.pt')

# 2. Tentukan gambar sampel yang akan dideteksi
# Kita menggunakan gambar bus kota yang disediakan oleh Ultralytics sebagai sampel online
url_sample = 'https://ultralytics.com/images/bus.jpg'

# 3. Jalankan inferensi
results = model(url_sample)

print("Proses deteksi selesai!")


Found https://ultralytics.com/images/bus.jpg locally at bus.jpg
image 1/1 c:\Users\EZZRA\Desktop\Transfer-Learning-Amarine-Vision\week5\bus.jpg: 640x480 4 persons, 1 bus, 1 stop sign, 136.1ms
Speed: 29.5ms preprocess, 136.1ms inference, 9.7ms postprocess per image at shape (1, 3, 640, 480)
Proses deteksi selesai!


## Langkah 4: Visualisasi & Menyimpan Hasil Prediksi

Objek `results` yang dikembalikan oleh model menyimpan banyak informasi seperti koordinat bounding box, confidence score, dan ID kelas. Kita bisa memvisualisasikannya secara langsung.

In [ ]:
# Menampilkan hasil deteksi di window baru (hanya bekerja jika dijalankan secara lokal)
# Jika dijalankan di Google Colab, gunakan matplotlib di bawah ini
results[0].show()

# Simpan hasil visualisasi dengan bounding box ke disk lokal
output_path = 'bus_prediction.jpg'
results[0].save(filename=output_path)

# Tampilkan gambar hasil prediksi di dalam notebook menggunakan Matplotlib
img_pred = cv.imread(output_path)
img_pred_rgb = cv.cvtColor(img_pred, cv.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(img_pred_rgb)
plt.axis('off')
plt.title("Hasil Deteksi YOLOv8 Pre-trained")
plt.show()

Mari kita bedah informasi koordinat bounding box mentah yang dihasilkan oleh model.

In [3]:
# Iterasi setiap bounding box hasil deteksi
for i, box in enumerate(results[0].boxes):
    cls_id = int(box.cls[0])           # ID Kelas numerik
    cls_name = model.names[cls_id]     # Nama kelas berdasarkan ID
    conf = float(box.conf[0])          # Confidence score (kepercayaan model)
    coords = box.xyxy[0].tolist()      # Koordinat bounding box [xmin, ymin, xmax, ymax]
    
    print(f"Objek {i+1}: {cls_name.upper()} | Confidence: {conf:.2f} | Box: {['{:.1f}'.format(coord) for coord in coords]}")

Objek 1: BUS | Confidence: 0.87 | Box: ['22.9', '231.3', '805.0', '756.8']
Objek 2: PERSON | Confidence: 0.87 | Box: ['48.6', '398.6', '245.3', '902.7']
Objek 3: PERSON | Confidence: 0.85 | Box: ['669.5', '392.2', '809.7', '877.0']
Objek 4: PERSON | Confidence: 0.83 | Box: ['221.5', '405.8', '345.0', '857.5']
Objek 5: PERSON | Confidence: 0.26 | Box: ['0.0', '550.5', '63.0', '873.4']
Objek 6: STOP SIGN | Confidence: 0.26 | Box: ['0.1', '254.5', '32.6', '324.9']


## Langkah 5: Deteksi Objek pada Video & Real-Time Webcam

YOLOv8 juga mendukung pemrosesan video dan streaming langsung dari webcam. Mari kita pelajari cara menerapkannya menggunakan OpenCV.

### 5a. Deteksi Objek pada File Video

Kita dapat membaca berkas video frame demi frame, mengumpankannya ke YOLOv8, dan menampilkan hasilnya secara live.

In [4]:
# Deteksi objek pada file video
import cv2 as cv
from ultralytics import YOLO

# Load model
model = YOLO('yolov8n.pt')

# Menggunakan berkas video lokal yang berisi objek umum (seperti orang) yang sudah ada di folder week4
video_path = '../week4/images/video.mp4'
cap = cv.VideoCapture(video_path)

# Verifikasi apakah video berhasil dibuka
if not cap.isOpened():
    print("Error: Gagal membuka berkas video.")
    print(f"Pastikan file video ada di path '{video_path}'.")
else:
    print("Memulai pemrosesan video... Tekan 'q' pada jendela window OpenCV untuk keluar lebih cepat.")
    
    # Batasi pemrosesan maks 150 frame agar demo tidak terlalu lama
    frame_count = 0
    max_frames = 150
    
    while cap.isOpened() and frame_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
            
        # Inferensi YOLOv8
        results = model(frame)
        
        # Gambar bounding box hasil prediksi
        annotated_frame = results[0].plot()
        
        # Tampilkan di jendela window
        cv.imshow('YOLOv8 Video Detection', annotated_frame)
        
        frame_count += 1
        
        # Tekan 'q' untuk keluar
        if cv.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv.destroyAllWindows()
    print(f"Pemrosesan video selesai. Total frame diproses: {frame_count}")

Memulai pemrosesan video... Tekan 'q' pada jendela window OpenCV untuk keluar lebih cepat.

0: 640x384 3 cars, 1 truck, 115.3ms
Speed: 4.1ms preprocess, 115.3ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 3 cars, 1 truck, 84.6ms
Speed: 4.6ms preprocess, 84.6ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 3 cars, 1 truck, 77.0ms
Speed: 3.3ms preprocess, 77.0ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 2 cars, 1 truck, 72.7ms
Speed: 2.2ms preprocess, 72.7ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 3 cars, 1 truck, 91.9ms
Speed: 2.6ms preprocess, 91.9ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 3 cars, 1 truck, 86.4ms
Speed: 3.0ms preprocess, 86.4ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 3 cars, 1 truck, 82.5ms
Speed: 5.3ms preprocess, 82.5ms inference, 1.1ms postprocess per image at

### 5b. Deteksi Objek Real-Time Menggunakan Webcam

Untuk mengaktifkan deteksi menggunakan kamera laptop Anda secara langsung, jalankan kode di bawah ini.

> **Catatan**: Sel di bawah ini hanya dapat berjalan dengan baik di laptop lokal Anda (tidak mendukung Google Colab cloud karena kendala akses webcam lokal).

In [5]:
# Deteksi objek real-time menggunakan Webcam
import cv2 as cv
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
cap = cv.VideoCapture(0) # 0 adalah indeks untuk webcam default

if not cap.isOpened():
    print("Error: Gagal membuka webcam laptop. Pastikan kamera tidak sedang digunakan oleh aplikasi lain.")
else:
    print("Webcam aktif! Tekan tombol 'q' pada jendela video untuk keluar.")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Gagal mengambil gambar dari kamera.")
            break
            
        # Gunakan stream=True untuk pemrosesan video streaming yang efisien secara memori
        results = model(frame, stream=True)
        
        for r in results:
            annotated_frame = r.plot()
            cv.imshow('YOLOv8 Live Webcam', annotated_frame)
            
        # Tekan 'q' untuk keluar
        if cv.waitKey(1) & 0xFF == ord('q'):
            break
    
    cap.release()
    cv.destroyAllWindows()
    print("Kamera telah dinonaktifkan.")

Webcam aktif! Tekan tombol 'q' pada jendela video untuk keluar.

0: 480x640 1 person, 1 laptop, 115.9ms
Speed: 3.6ms preprocess, 115.9ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 laptop, 103.6ms
Speed: 3.5ms preprocess, 103.6ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 1 laptop, 100.1ms
Speed: 3.6ms preprocess, 100.1ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 1 laptop, 101.1ms
Speed: 2.6ms preprocess, 101.1ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 laptop, 103.6ms
Speed: 2.4ms preprocess, 103.6ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 laptop, 94.2ms
Speed: 2.4ms preprocess, 94.2ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 1 laptop, 95.2ms
Speed: 4.8ms preprocess, 95.2ms inference, 1.3ms postprocess per imag

## Langkah 6: Integrasi Dataset Roboflow

Setelah Anda menyelesaikan tugas mandiri membuat project, mengunggah gambar, menggambar bounding box, dan menerapkan augmentasi di Roboflow, Anda dapat mengekspor dataset tersebut ke format **YOLOv8 PyTorch**.

Roboflow akan menyediakan potongan kode Python unik untuk mengunduh dataset secara terprogram. Kode tersebut memiliki format seperti di bawah ini.

> **Perhatian**: Jangan bagikan API Key Anda secara publik ke GitHub demi keamanan akun Anda.

In [6]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="q294SmqNM6eLolt7AeUe")
project = rf.workspace("visionamarine").project("reels-moi4j")
version = project.version(1)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to reels-1 in yolov8:: 100%|██████████| 2344/2344 [00:03<00:00, 646.04it/s]


## Langkah 7: Pemeriksaan Struktur File Dataset

Setelah dataset berhasil terunduh, folder dataset baru (misalnya bernama `Amarine-Vision-1` atau sesuai nama project Anda) akan terbentuk di direktori ini. Mari kita buat script pembaca file `data.yaml` untuk memastikan semuanya telah terstruktur dengan benar.

In [11]:
import yaml

# Ubah nama folder di bawah ini sesuai dengan nama folder dataset hasil unduhan Roboflow Anda
folder_dataset = 'reels-1'
path_yaml = os.path.join(folder_dataset, 'data.yaml')

if os.path.exists(path_yaml):
    with open(path_yaml, 'r') as file:
        data_config = yaml.safe_load(file)
        print("--- KONFIGURASI DATASET --- ")
        print(yaml.dump(data_config, default_flow_style=False))
        
        # Periksa jalur train & val
        print("Jalur Data Training:", data_config.get('train'))
        print("Jalur Data Validasi:", data_config.get('val'))
        print("Daftar Kelas:", data_config.get('names'))
else:
    print(f"File '{path_yaml}' belum ditemukan.")
    print("Langkah ini dapat dilakukan setelah Anda mengunduh dataset kustom Anda di Langkah 6.")

--- KONFIGURASI DATASET --- 
names:
- gate
- pole_gate
- upper_gate
- upper_left_gate
nc: 4
roboflow:
  license: CC BY 4.0
  project: reels-moi4j
  url: https://universe.roboflow.com/visionamarine/reels-moi4j/dataset/1
  version: 1
  workspace: visionamarine
test: ../test/images
train: ../train/images
val: ../valid/images

Jalur Data Training: ../train/images
Jalur Data Validasi: ../valid/images
Daftar Kelas: ['gate', 'pole_gate', 'upper_gate', 'upper_left_gate']


## Langkah 8: Pengujian Video Menggunakan Model Hasil Training Kustom (Testing Model)

Setelah Anda melatih model YOLOv8 pada dataset kustom Anda di **Week 6**, Anda akan mendapatkan file bobot terbaik bernama `best.pt` yang tersimpan di dalam folder `runs/detect/train/weights/`.

Untuk mengujinya pada video testing kustom Anda, jalankan sel di bawah ini (sel ini dapat digunakan setelah Anda menyelesaikan pelatihan model kustom Anda di pertemuan berikutnya).

In [14]:
# Jalankan deteksi video menggunakan model kustom hasil training
import cv2 as cv
from ultralytics import YOLO
import os

# 1. Lokasi file best.pt hasil training Anda nanti
custom_model_path = 'best.pt'

# 2. Tentukan video testing Anda
# Ganti dengan path berkas video asli milik Anda, misalnya: 'images/marine_test_video.mp4'
video_test_path = 'videoplayback.mp4'


# Jalankan deteksi video jika file weights kustom hasil training sudah ada
if os.path.exists(custom_model_path):
    # Muat model kustom
    model_custom = YOLO(custom_model_path)
    
    # Buka video
    cap = cv.VideoCapture(video_test_path)
    
    if not cap.isOpened():
        print(f"Error: Gagal membuka berkas video testing '{video_test_path}'.")
    else:
        print("Memulai deteksi video dengan model kustom... Tekan 'q' untuk keluar.")
        
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
                
            # Inferensi frame dengan model kustom Anda
            results = model_custom(frame)
            
            # Terapkan anotasi bounding box kelas kustom Anda
            annotated_frame = results[0].plot()
            
            # Tampilkan video hasil deteksi
            cv.imshow('Custom Model Detection', annotated_frame)
            
            if cv.waitKey(1) & 0xFF == ord('q'):
                break
                
        cap.release()
        cv.destroyAllWindows()
        print("Pengujian video selesai.")
else:
    print(f"Error: File '{custom_model_path}' masih belum ditemukan setelah training.")

Memulai deteksi video dengan model kustom... Tekan 'q' untuk keluar.

0: 416x640 2 gates, 133.2ms
Speed: 95.8ms preprocess, 133.2ms inference, 1.1ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 gate, 79.5ms
Speed: 3.8ms preprocess, 79.5ms inference, 1.0ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 gate, 90.9ms
Speed: 3.7ms preprocess, 90.9ms inference, 1.1ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 gate, 89.9ms
Speed: 3.4ms preprocess, 89.9ms inference, 1.3ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 gate, 82.4ms
Speed: 3.2ms preprocess, 82.4ms inference, 1.1ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 gate, 77.4ms
Speed: 2.3ms preprocess, 77.4ms inference, 1.5ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 gate, 76.2ms
Speed: 3.6ms preprocess, 76.2ms inference, 1.1ms postprocess per image at shape (1, 3, 416, 640)

0: 416x640 1 gate, 72.4ms
Speed: 3.0ms preprocess, 72.4ms 

## Langkah 9: Evaluasi Kuantitatif Metrik Performa Model (Precision, Recall, mAP)

Selain melihat hasil deteksi secara visual pada video (Langkah 8), kita perlu mengukur akurasi model secara ilmiah menggunakan metrik evaluasi deteksi objek standar (mAP, Precision, Recall).

Fungsi `model.val()` pada YOLOv8 secara otomatis akan menghitung performa model Anda pada data validasi yang tertera di dalam berkas konfigurasi `data.yaml`.

In [ ]:
# Jalankan evaluasi kuantitatif pada model kustom Anda
from ultralytics import YOLO
import os

custom_model_path = 'runs/detect/train/weights/best.pt'

if not os.path.exists(custom_model_path):
    print(f"File weights kustom '{custom_model_path}' belum ditemukan.")
    print("Sel evaluasi metrik ini siap digunakan setelah Anda melakukan training model di Week 6!")
else:
    # 1. Muat model kustom Anda
    model_custom = YOLO(custom_model_path)
    
    # 2. Jalankan evaluasi data validasi
    print("Memulai kalkulasi metrik evaluasi pada dataset validasi...")
    metrics = model_custom.val()
    
    # 3. Cetak metrik penting
    print("\n--- HASIL EVALUASI METRIK --- ")
    print(f"Precision (Presisi)                          : {metrics.box.mp:.4f}")
    print(f"Recall (Sensitivitas)                         : {metrics.box.mr:.4f}")
    print(f"Mean Average Precision @ IoU=0.5 (mAP50)       : {metrics.box.map50:.4f}")
    print(f"Mean Average Precision @ IoU=0.5:0.95 (mAP50-95): {metrics.box.map:.4f}")

--- 

### 🌟 Apa Langkah Selanjutnya?
Jika langkah-langkah di atas berjalan dengan lancar, Anda telah berhasil:
1. Melakukan deteksi objek awal menggunakan YOLOv8 model pre-trained (Gambar, Video, Webcam).
2. Memahami cara menghubungkan cloud dataset dari Roboflow ke kode Python.

Pada **Week 6**, kita akan menggunakan dataset kustom yang sudah Anda persiapkan minggu ini untuk melatih (*training*) model YOLOv8 kustom kita agar mampu mendeteksi objek spesifik bawah air/kelautan secara akurat! 🚀